# FDCA Part II — Walkthrough paso a paso

Recorrido sección por sección del ATBD 3.4.2.14 a 3.4.2.18, verificando que
`part2.py` implemente cada paso correctamente sobre los candidatos que
salieron de Part I.

La estructura sigue las secciones ATBD 3.4.2.14 → 3.4.2.18:

| Sección | Qué hace |
|---------|----------|
| 3.4.2.14 |  |
| 3.4.2.15 |  |
| 3.4.2.16 |  |
| 3.4.2.17 |  |

## 0. Setup

In [7]:
import os
import sys
import subprocess
from pathlib import Path

try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

BRANCH = "testing-fdca-real"

if IN_COLAB:
    print("Ejecutando en Google Colab")

    GH_TOKEN = userdata.get("GITHUB_TOKEN")
    REPO_URL = f"https://{GH_TOKEN}@github.com/Vale23-23/GERIS-PFC.git"
    CLONE_DIR = "/content/GERIS-PFC"

    if not os.path.exists(CLONE_DIR):
        result = subprocess.run(
            ["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, CLONE_DIR],
            capture_output=True, text=True,
        )
        if result.returncode != 0:
            raise RuntimeError(result.stderr)
        print("Repo clonado OK")
    else:
        pull = subprocess.run(
            ["git", "-C", CLONE_DIR, "pull"],
            capture_output=True, text=True,
        )
        print("Pull OK")

    PROJECT_ROOT = Path(CLONE_DIR) / "implementacion"
else:
    print("Ejecutando localmente (VS Code u otro IDE)")
    cwd = Path.cwd()
    if (cwd / "implementacion" / "fdca").exists():
        PROJECT_ROOT = cwd / "implementacion"
    elif (cwd / "fdca").exists():
        PROJECT_ROOT = cwd
    else:
        PROJECT_ROOT = cwd

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
sys.path.insert(0, str(PROJECT_ROOT))

Ejecutando localmente (VS Code u otro IDE)
PROJECT_ROOT = /home/luciana/Escritorio/Fing/PFC/GERIS-PFC/implementacion


In [8]:
# ── Celda 2: bajar datos desde Hugging Face ───────────────────────────────
import os

try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    HF_TOKEN = userdata.get("HF_TOKEN")
    DATA_ROOT = "/content/data"
else:
    from dotenv import load_dotenv

    env_candidates = [Path.cwd() / ".env", Path.cwd().parent / ".env"]
    for env_path in env_candidates:
        if env_path.exists():
            load_dotenv(env_path)
            break

    HF_TOKEN = os.getenv("HF_TOKEN")
    DATA_ROOT = Path.cwd() / "data"

from huggingface_hub import snapshot_download

HF_REPO_ID = "valentina2323/GERIS-Goes19-uruguay-fires"
TIMESTAMP = "20250926_1900"   # ajustar al timestamp que quieras walkthrough-ear
LOCAL_DATA = os.path.join(DATA_ROOT, TIMESTAMP)

os.makedirs(LOCAL_DATA, exist_ok=True)

snapshot_download(
    repo_id=HF_REPO_ID,
    repo_type="dataset",
    token=HF_TOKEN,
    allow_patterns=[
        f"uruguay/*/{TIMESTAMP}.npy",
        "uruguay/**/*.json",
        "uruguay/camel_emissivity/*.nc",
    ],
    local_dir=str(DATA_ROOT),
)

print(f"Datos en: {LOCAL_DATA}")

/home/luciana/miniconda3/envs/geris/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching ... files: 23it [00:00, 76139.69it/s]

Datos en: /home/luciana/Escritorio/Fing/PFC/GERIS-PFC/implementacion/data/20250926_1900


In [9]:
import sys
import numpy as np
sys.path.insert(0, str(PROJECT_ROOT))

from fdca.fdca_adapter import load_fdca_input

CONFIG_PATH = str(PROJECT_ROOT / "fdca" / "config.yaml")
DATASET_ROOT = str(DATA_ROOT)

inp = load_fdca_input(
    timestamp=TIMESTAMP,
    region="uruguay",
    config_path=CONFIG_PATH,
    dataset_root=DATASET_ROOT,
    verbose=True,
)

print(f"Grilla: {inp.bt7.shape}  |  BT7 [{np.nanmin(inp.bt7):.1f}, {np.nanmax(inp.bt7):.1f}] K")

  ABI-L1b-Rad-B07       : shape=(224, 303)  range=[0.58, 25.59]
  ABI-L1b-Rad-B14       : shape=(224, 303)  range=[75.79, 196.25]
  ABI-L1b-Rad-B13       : shape=(224, 303)  range=[66.67, 177.84]
  ABI-L1b-Rad-B15       : shape=(224, 303)  range=[75.24, 181.48]
  ABI-L1b-Rad-B02       : shape=(893, 1212)  range=[12.06, 344.00]

  SZA range [°]         : 48.0 – 55.9
  LZA range [°]         : 31.3 – 40.1
  Píxeles diurnos       : 100%

  BT7 range [K]         : 288.9 – 410.9
  BT14 range [K]        : 271.9 – 339.2
  refl2 range           : 0.051 – 0.352
  Emisividad            : CAMEL V3 clim. (CAM5K30EMCLIM_emis_climatology_09Month_V003.nc)
  TPW range [mm]        : 25.0 – 35.0  (estimación climatológica)

  ✓ FDCAInput construido — shape (224, 303)
Grilla: (224, 303)  |  BT7 [288.9, 410.9] K


In [10]:
from fdca.part1 import run_part1

fire_mask, fail_char_arr, candidates = run_part1(
    bt7=inp.bt7, rad7=inp.rad7, bt14=inp.bt14, rad14=inp.rad14,
    bt13=inp.bt13, rad13=inp.rad13, bt15=inp.bt15, refl2=inp.refl2,
    latitudes=inp.latitudes, longitudes=inp.longitudes,
    sza=inp.sza, glint_angle=inp.glint_angle, lza=inp.lza, azimuth=inp.azimuth,
    tpw=inp.tpw, emiss7=inp.emiss7, emiss14=inp.emiss14,
    lut_tpw=inp.lut_tpw, FPT=inp.FPT,
    coeffs7=inp.coeffs7, coeffs14=inp.coeffs14, coeffs13=inp.coeffs13,
    land_cover=inp.land_cover, land_mask=inp.land_mask,
    desert_mask=inp.desert_mask, usgs_eco=inp.usgs_eco,
    data_quality=inp.data_quality,
)

print(f"Candidatos generados por Part I: {len(candidates)}")

Candidatos generados por Part I: 543


In [11]:
%reload_ext autoreload
%autoreload 2

import copy
import pandas as pd
from datetime import datetime

from fdca.part2 import (
    run_part2,
    _eliminate_false_alarm, _reassign_cloud_glint_edge,
    _reassign_fog_edge, _upgrade_confidence,
    _assign_fire_category, _std_reflb_part2, _refl_along_scan_part2,
)
from fdca.constants import FailChar, FireMask, BKG_MAX_ITER, TEMPORAL_WINDOW_H, TEMPORAL_PIXEL_RAD

# prev_fire_mask / current_epoch para el filtro temporal (3.4.2.16).
# Sin corrida previa disponible todavía → desactivamos el filtro temporal.
prev_fire_mask = None
EPOCH_2001 = datetime(2001, 1, 1)
current_epoch = (datetime.strptime(TIMESTAMP, "%Y%m%d_%H%M") - EPOCH_2001).total_seconds()

df_p2 = pd.DataFrame([vars(c) for c in candidates])
df_p2[["i","j","bt7","bt7_bkg","bt14","bt7_corr","refl_pixel","reflb",
       "fail_char","n_passes","fire_temp"]].head()

,i,j,bt7,bt7_bkg,bt14,bt7_corr,refl_pixel,reflb,fail_char,n_passes,fire_temp
0,1,145,315.888519,309.995544,308.052460,317.640953,0.443790,0.264687,9,0,-367.779965
1,1,146,315.961975,310.013977,308.133698,317.795107,0.444424,0.257394,9,0,-368.352200
2,4,289,309.632141,304.181519,304.037140,314.034708,0.268672,0.130961,6,0,-999.000000
3,5,288,309.869659,303.862793,304.427551,314.480200,0.263993,0.124235,9,0,-999.000000
4,5,289,310.339355,304.040649,304.955536,314.971198,0.265415,0.126749,9,0,-999.000000


## 3.4.2.14 — Start Part II: Threshold test

Tres tests OR sobre BT3.9 vs. background/BT14, cada uno combinado con el
check de Refl/along-scan (ATBD pág. 38-39). Si cualquiera da `True`, el
candidato se descarta como falsa alarma antes de seguir.

In [12]:
resultados_314 = [_eliminate_false_alarm(c) for c in candidates]
df_p2["eliminado_314"], df_p2["razon_314"] = zip(*resultados_314)

n_elim = int(df_p2["eliminado_314"].sum())
print(f"Eliminados por umbral Part II (3.4.2.14, tests 1-3): {n_elim} / {len(df_p2)}")
df_p2["razon_314"].value_counts()

Eliminados por umbral Part II (3.4.2.14, tests 1-3): 0 / 543


razon_314
    543
Name: count, dtype: int64

### 3.4.2.14 (cont.) — Re-evaluación glint / borde de nube

Solo para candidatos que sobrevivieron el filtro anterior y traían flag
F9 o F10 de Part I. Si el albedo indica borde de nube/glint y BT3.9
corregida está por debajo de umbral, se reasigna a F11.

In [13]:
sobrevivientes = [c for c in candidates if not _eliminate_false_alarm(c)[0]]
sza_cos_sob = np.cos(np.radians([c.sza for c in sobrevivientes]))
day_off_sob = np.where([c.is_day for c in sobrevivientes], sza_cos_sob * 20.0, 0.0)

reasignados_edge = [
    _reassign_cloud_glint_edge(c, do) for c, do in zip(sobrevivientes, day_off_sob)
]

n_f9_f10 = sum(1 for c in sobrevivientes if c.fail_char in (FailChar.F9, FailChar.F10))
print(f"Candidatos con flag F9/F10 elegibles: {n_f9_f10}")
print(f"Reasignados a F11 (borde nube/glint):  {sum(reasignados_edge)}")

Candidatos con flag F9/F10 elegibles: 362
Reasignados a F11 (borde nube/glint):  0


## 3.4.2.15 — Determine fire category

Categoriza cada candidato (10/30 procesado, 11/31 saturado, 12/32
nube/humo, 13-15/33-35 alta/media/baja probabilidad). Antes de la
categorización, un segundo test de borde-nube (distinto al de 3.4.2.14,
sobre BT3.9-BT11.2 del background en vez de albedo) puede reasignar a
F11. Luego, los candidatos F3/F4/F6/F8 sin solución Dozier válida
(fire_temp < 0) pueden subir a alta/media confianza.

In [14]:
# segundo test de borde de nube

sza_cos_all = np.cos(np.radians(df_p2["sza"].values))

reasignados_315 = [_reassign_fog_edge(c, sc) for c, sc in zip(candidates, sza_cos_all)]
print(f"Reasignados a F11 por fog_c1/fog_c2 (3.4.2.15): {sum(reasignados_315)}")

Reasignados a F11 por fog_c1/fog_c2 (3.4.2.15): 0


In [15]:
# categorización + upgrade

categorias_base = [_assign_fire_category(c, fire_mask) for c in candidates]
upgrades = [_upgrade_confidence(c) for c in candidates]

df_p2["categoria_base"]  = categorias_base
df_p2["upgrade_code"]    = [u[0] for u in upgrades]
df_p2["upgrade_delta"]   = [u[1] for u in upgrades]
df_p2["categoria_final"] = [
    u[0] if u[0] is not None else base
    for base, u in zip(categorias_base, upgrades)
]

df_p2["categoria_final"].value_counts().sort_index()

categoria_final
10     47
12    362
15    134
Name: count, dtype: int64

## 3.4.2.16 — Temporal filtering

Compara la máscara de fuego previa (segundos desde 1-ene-2001 del último
fuego detectado en cada píxel) contra un radio de 1 línea/elemento. Si
hubo fuego dentro de las últimas `TEMPORAL_WINDOW_H` horas en ese entorno,
el código de fuego se corre +20 (10→30, 11→31, etc.).

In [16]:
from fdca.constants import TEMPORAL_WINDOW_H, TEMPORAL_PIXEL_RAD

def _check_temporal_filter(cand, prev_fire_mask, current_epoch):
    if prev_fire_mask is None:
        return False
    L, W = prev_fire_mask.shape
    for di in range(-TEMPORAL_PIXEL_RAD, TEMPORAL_PIXEL_RAD + 1):
        for dj in range(-TEMPORAL_PIXEL_RAD, TEMPORAL_PIXEL_RAD + 1):
            ni, nj = cand.i + di, cand.j + dj
            if 0 <= ni < L and 0 <= nj < W:
                last_t = prev_fire_mask[ni, nj]
                if last_t > 0 and (current_epoch - last_t) <= TEMPORAL_WINDOW_H * 3600.0:
                    return True
    return False

# prev_fire_mask, current_epoch deben estar definidos (o None / 0.0 si no hay
# corrida previa disponible todavía — desactiva el filtro temporal)
df_p2["filtrado_temporal"] = [
    _check_temporal_filter(c, prev_fire_mask, current_epoch) for c in candidates
]
print(f"Candidatos temporalmente filtrados: {int(df_p2['filtrado_temporal'].sum())}")

Candidatos temporalmente filtrados: 0


## 3.4.2.17 — Fire Output / 3.4.2.18 — End Part II

Sin lógica propia: son la descripción de la salida final (Tabla 3.10:
fire mask codes, subpixel size/temp/FRP, previous fire mask, QA flags,
metadata) y el cierre del algoritmo. Se verifica acá que la salida de
`run_part2` tenga la forma esperada.

In [18]:
print(type(FireMask))
print(FireMask.__mro__ if hasattr(FireMask, "__mro__") else "sin __mro__")

<class 'type'>
(<class 'fdca.constants.FireMask'>, <class 'object'>)


In [19]:
# Copias para no pisar el estado usado en las celdas de arriba
candidates_run = copy.deepcopy(candidates)
fire_mask_run  = fire_mask.copy()
fail_char_run  = fail_char_arr.copy()

fire_mask_out, fail_char_out, confirmed = run_part2(
    candidates_run, fire_mask_run, fail_char_run,
    prev_fire_mask, current_epoch,
)

print(f"Candidatos confirmados como fuego: {len(confirmed)} / {len(candidates)}")

codigos, cuentas = np.unique(fire_mask_out[fire_mask_out >= 10], return_counts=True)

code_to_name = {
    v: k for k, v in vars(FireMask).items()
    if not k.startswith("_") and isinstance(v, int)
}

for cod, cnt in zip(codigos, cuentas):
    nombre = code_to_name.get(int(cod), str(int(cod)))
    print(f"  código {int(cod):>3}  {nombre:<20}  {cnt} píxeles")

Candidatos confirmados como fuego: 543 / 543
  código  10  PROCESSED             47 píxeles
  código  12  CLOUD_CONTAM          362 píxeles
  código  15  LOW_PROB              134 píxeles
  código 100  FIRE_FREE             67235 píxeles
  código 215  CLOUD_ALBEDO          94 píxeles
